In [ ]:
import os
os.makedirs("Figs", exist_ok=True)  # figures are written to ./Figs

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# ---- BMRB statistics (for the NOREF 4-sigma outlier filter; same as rmse_comparison_4pub) ----
_AA_1TO3 = {
    'A': 'ALA', 'R': 'ARG', 'N': 'ASN', 'D': 'ASP', 'C': 'CYS', 'Q': 'GLN',
    'E': 'GLU', 'G': 'GLY', 'H': 'HIS', 'I': 'ILE', 'L': 'LEU', 'K': 'LYS',
    'M': 'MET', 'F': 'PHE', 'P': 'PRO', 'S': 'SER', 'T': 'THR', 'W': 'TRP',
    'Y': 'TYR', 'V': 'VAL',
}

def set_style(base=12, family="Arial"):
    plt.rcParams.update({
        "font.family": family,
        "font.size": base,
        "axes.titlesize": base + 2,
        "axes.labelsize": base,
        "xtick.labelsize": base - 2,
        "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2,
    })

set_style(base=14)

labels = {"noref": "rBMRB-BASELINE", "panav": "rBMRB-PANAV", "lacs": "rBMRB-LACS"}
USE_NMASK = {"noref": True, "panav": True, "lacs": True}   # True: use the shipped N_mask; False: 4-sigma filter

# train / val / test of the three training sets (same proteins, only the N referencing differs)
train_paths = {
    "noref": "../../../data/rBMRB/cluster_30p/NOREF/train.json",
    "panav": "../../../data/rBMRB/cluster_30p/PANAV/train.json",
    "lacs":  "../../../data/rBMRB/cluster_30p/LACS/train.json",
}
val_paths = {
    "noref": "../../../data/rBMRB/cluster_30p/NOREF/val.json",
    "panav": "../../../data/rBMRB/cluster_30p/PANAV/val.json",
    "lacs":  "../../../data/rBMRB/cluster_30p/LACS/val.json",
}
test_paths = {
    "noref": "../../../data/rBMRB/cluster_30p/NOREF/test.json",
    "panav": "../../../data/rBMRB/cluster_30p/PANAV/test.json",
    "lacs":  "../../../data/rBMRB/cluster_30p/LACS/test.json",
}

# ---- two extra masks: tag/linker mask (train/val/test) + PANAV flag mask (val/test only) ----
TAG_LABELS_CSV   = '../../../data/rBMRB/tag_labels.csv'
PANAV_FLAGS_JSON = '../../../data/rBMRB/bmrb_cs_data_withoffset_panav_v2_flags_only.json'

def load_tag_mask_dict(csv_path, id_col='BMRB_entry', label_col='label'):
    """pdb_id -> keep-mask. Labels 'A'/'P' are real residues; 'T'/'t' are expression tags / linkers and are excluded."""
    df = pd.read_csv(csv_path)
    out = {}
    for _, row in df.iterrows():
        label = row.get(label_col)
        if not isinstance(label, str) or not label:
            continue
        pdb_id = str(row[id_col]).lower().removeprefix('bmr')
        out[pdb_id] = np.array([c in ('A', 'P') for c in label], dtype=bool)
    return out

def load_flag_mask_dict(json_path, atom='N'):
    """pdb_id -> keep-mask. In the raw '{atom}_flag' True means exclude; inverted here so True = keep."""
    with open(json_path) as f:
        rbmrb = json.load(f)
    flag_key = f'{atom}_flag'
    out = {}
    for ref_id, flag_lst in rbmrb[flag_key].items():
        pdb_id = rbmrb['entry_ID'][ref_id].lower().removeprefix('bmr')
        out[pdb_id] = np.array([not bool(x) for x in flag_lst], dtype=bool)
    return out

tag_mask_dict  = load_tag_mask_dict(TAG_LABELS_CSV)
flag_mask_dict = load_flag_mask_dict(PANAV_FLAGS_JSON, atom='N')


def collect_N(path, use_nmask, apply_flag_mask):
    """Collect filtered N chemical shifts.
    use_nmask=True : PANAV/LACS -- proline + None + the shipped N_mask
    use_nmask=False: NOREF      -- proline + None + BMRB 4-sigma outlier filter
    The tag mask (expression-tag residues) is applied to all three splits;
    with apply_flag_mask=True the PANAV flag mask is added as well (val/test only, not train).
    Entries without a tag/flag mask are not filtered by it; if the length differs only the overlapping prefix is trusted
    and the tail is excluded (same "no info means don't gate" rule as the trainer).
    """
    d = json.load(open(path))
    out = []
    raw = 0
    for key in d["N"]:
        seq    = d["sequence"][key]
        shifts = np.array([s if s is not None else np.nan for s in d["N"][key]], float)
        raw += int(np.sum(~np.isnan(shifts)))
        n = len(seq)

        pro_mask  = np.array([res != "P" for res in seq])
        none_mask = ~np.isnan(shifts)

        if use_nmask:
            extra_mask = np.array(d["N_mask"][key], dtype=bool)

        pdb_id = d["entry_ID"][key].lower().removeprefix("bmr")

        tag_mask = tag_mask_dict.get(pdb_id)
        if tag_mask is not None:
            m = min(n, len(tag_mask))
            gated = np.zeros(n, dtype=bool)
            gated[:m] = tag_mask[:m]
            extra_mask = extra_mask & gated

        if apply_flag_mask:
            flag_mask = flag_mask_dict.get(pdb_id)
            if flag_mask is not None:
                m = min(n, len(flag_mask))
                gated = np.zeros(n, dtype=bool)
                gated[:m] = flag_mask[:m]
                extra_mask = extra_mask & gated

        mask = pro_mask & none_mask & extra_mask
        out.append(shifts[mask])

    return np.concatenate(out), raw


# ---- filter each split, then combine ----
marg, raw_counts = {}, {}
for k in ["noref", "panav", "lacs"]:
    parts, raw_total = [], 0
    for paths, apply_flag in ((train_paths, False), (val_paths, True), (test_paths, True)):
        m, r = collect_N(paths[k], use_nmask=USE_NMASK[k], apply_flag_mask=apply_flag)
        parts.append(m)
        raw_total += r
    marg[k] = np.concatenate(parts)
    raw_counts[k] = raw_total

In [ ]:
colors = {"noref": "#999999", "panav": "#61AFEF", "lacs": "#E06C75"}
# ---------- plot: KDE curves ----------
fig, ax = plt.subplots(figsize=(8, 5))

xs = np.linspace(90, 150, 600)
for k in ["noref", "panav", "lacs"]:
    kde = gaussian_kde(marg[k], bw_method=0.15)   # smaller bw follows detail, larger bw is smoother
    ys = kde(xs)
    ax.plot(xs, ys, lw=2, color=colors[k], 
            label=f"{labels[k]}\n(μ={marg[k].mean():.2f}, σ={marg[k].std():.2f})")

ax.set_xlabel("$^{15}$N chemical shift (ppm)")
ax.set_ylabel("Density")
ax.set_xlim(95, 145)
ax.invert_xaxis()
ax.set_ylim(bottom=0)
ax.legend(frameon=False, loc="upper right")
# ax.grid(alpha=0.3, linestyle="--")

plt.tight_layout()
plt.savefig("Figs/dataset_distribution_acrossmodel.png", dpi=300, bbox_inches="tight")
plt.savefig("Figs/dataset_distribution_acrossmodel.svg", bbox_inches="tight")
plt.show()

for k in ["noref", "panav", "lacs"]:
    print(f"{labels[k]:<6} kept {marg[k].size:>6} / {raw_counts[k]:>6} non-None N  "
          f"(removed {100*(1-marg[k].size/raw_counts[k]):.1f}%)  μ={marg[k].mean():.3f} σ={marg[k].std():.3f}")